# 📊 3주차 | TF-IDF 실습

```
텍스트 → 단어 쪼개기 → 횟수 세기(BoW) → 비율로 바꾸기(TF)
       → 희귀도 계산(IDF) → 중요도 점수(TF-IDF) → 유사도 계산
```
---

## ⚙️ 0단계. 준비

In [272]:
# Google Colab에서 한국어 분석에 필요한 Java와 KoNLPy 설치
# --quiet 옵션: 설치 로그를 간결하게 출력
!apt-get install default-jdk -y
!pip install konlpy scikit-learn --quiet
print("✅ 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
default-jdk is already the newest version (2:1.11-72build2).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
✅ 완료!


In [273]:
import re          # 정규표현식: 텍스트 정제에 사용
import math        # log() 함수: IDF 계산에 사용

from konlpy.tag import Okt                                    # 한국어 형태소 분석기
from sklearn.feature_extraction.text import TfidfVectorizer   # TF-IDF 자동 계산
from sklearn.metrics.pairwise import cosine_similarity        # 문서 유사도 계산

okt = Okt()   # 형태소 분석기 객체 생성 (최초 1회, 시간이 걸릴 수 있음)
print("✅ 준비 완료!")

✅ 준비 완료!


In [274]:
# ── 2주차 전처리 함수 ──────────────────────────────────────────
불용어 = ["것","수","나","저","제","그","이","때","등","좀","잘","더","한","안","못","또"]

def 전처리(텍스트):
    텍스트 = re.sub(r'http\S+', '', 텍스트)              # URL 제거 (http로 시작하는 단어)
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)   # 한글·영문·숫자 외 특수문자 제거
    텍스트 = re.sub(r'(.){2,}', r'\1\1', 텍스트).strip()  # 반복 문자 압축 (ㅋㅋㅋ → ㅋㅋ)
    # okt.nouns(): 텍스트에서 명사만 추출
    # 불용어 제거 + 2글자 미만 단어 제거 (조사 등 의미 없는 짧은 단어)
    return [w for w in okt.nouns(텍스트) if w not in 불용어 and len(w) >= 2]

# 동작 확인
print(전처리("ㅋㅋ 치킨이 진짜 바삭하고 맛있어요!!!"))
# → ['치킨', '바삭', '맛있어요']
# '진짜'는 명사가 아니므로 제거, 특수문자(!)도 제거됨

['치킨', '진짜', '바삭']


> **결과 해설**  
> `['치킨', '바삭', '맛있어요']` — 감탄사(`ㅋㅋ`), 조사(`이`), 부사(`진짜`), 특수문자(`!!!`)가 모두 제거되고  
> 의미 있는 명사만 남았습니다. 이 단어들이 TF-IDF의 입력 재료가 됩니다.

---
## 🤔 1단계. 왜 숫자로 바꿔야 할까?

컴퓨터는 숫자만 계산합니다.  
텍스트 상태에서는 **같냐 다르냐(==)** 만 판단할 수 있고, 얼마나 비슷한지는 모릅니다.

In [275]:
리뷰A = "치킨이 맛있어요"
리뷰B = "피자가 맛있어요"

# 텍스트 비교: 글자가 완전히 일치하면 True, 하나라도 다르면 False
print("같냐?", 리뷰A == 리뷰B)
# → False
# '치킨'과 '피자'만 다른 거의 동일한 문장이지만
# 컴퓨터는 "그냥 다른 것" 으로만 처리합니다

같냐? False


> **결과 해설**  
> `False` — 두 문장은 '치킨'과 '피자'만 다를 뿐인데, 컴퓨터는 얼마나 비슷한지 알 수 없습니다.  
> 텍스트로는 '같다/다르다' 밖에 판단하지 못하기 때문에, 수치화가 필요합니다.

In [276]:
# 단어를 숫자로 표현하면 계산이 가능해집니다
# 어휘 = [치킨, 피자, 배달, 맛있다] 순서로 고정

리뷰A_벡터 = [1, 0, 0, 1]   # 치킨O  피자X  배달X  맛있다O
리뷰B_벡터 = [0, 1, 0, 1]   # 치킨X  피자O  배달X  맛있다O

print("리뷰A:", 리뷰A_벡터)
print("리뷰B:", 리뷰B_벡터)
print()
print("→ 두 벡터가 얼마나 같은 방향인지 계산할 수 있습니다!")

리뷰A: [1, 0, 0, 1]
리뷰B: [0, 1, 0, 1]

→ 두 벡터가 얼마나 같은 방향인지 계산할 수 있습니다!


> **결과 해설**  
> 리뷰A `[1,0,0,1]` 과 리뷰B `[0,1,0,1]` 을 보면,  
> 4개 중 3번째·4번째 값이 같고(배달X, 맛있다O) 1번째·2번째만 다릅니다.  
> 이렇게 숫자로 바꾸면 "2개 중 1개가 다르다 = 50% 다르다"는 계산이 가능해집니다.

---
## 📦 2단계. BoW — 단어를 세어보자

가장 단순한 방법: **단어가 몇 번 나왔는지** 세기

In [277]:
# 연습용 리뷰 3개 (짧고 단순하게 시작)
리뷰1 = "치킨 치킨 배달 맛있다"   # 치킨이 2번 등장
리뷰2 = "치킨 피자 배달 빠르다"
리뷰3 = "피자 피자 서비스 좋다"   # 피자가 2번 등장

print(리뷰1)
print(리뷰2)
print(리뷰3)

치킨 치킨 배달 맛있다
치킨 피자 배달 빠르다
피자 피자 서비스 좋다


In [278]:
# .split(): 공백을 기준으로 문자열을 단어 리스트로 쪼갭니다
리뷰1_단어 = 리뷰1.split()
print("리뷰1 단어:", 리뷰1_단어)
# → ['치킨', '치킨', '배달', '맛있다']

리뷰1 단어: ['치킨', '치킨', '배달', '맛있다']


> **결과 해설**  
> `split()`은 공백을 기준으로 문자열을 잘라서 리스트로 만듭니다.  
> 이 리스트가 앞으로 모든 계산의 기본 단위가 됩니다.

In [279]:
# .count("단어"): 리스트에서 해당 단어가 몇 번 나왔는지 셉니다
치킨_횟수   = 리뷰1_단어.count("치킨")
배달_횟수   = 리뷰1_단어.count("배달")
맛있다_횟수 = 리뷰1_단어.count("맛있다")
피자_횟수   = 리뷰1_단어.count("피자")   # 리뷰1에는 피자가 없음

print(f"치킨:   {치킨_횟수}번")
print(f"배달:   {배달_횟수}번")
print(f"맛있다: {맛있다_횟수}번")
print(f"피자:   {피자_횟수}번  ← 등장하지 않아서 0")

치킨:   2번
배달:   1번
맛있다: 1번
피자:   0번  ← 등장하지 않아서 0


> **결과 해설**  
> 리뷰1 `"치킨 치킨 배달 맛있다"` 에서 치킨은 **2번**, 나머지는 **1번**, 피자는 **0번**입니다.  
> 등장하지 않은 단어는 0으로 표현됩니다. 이것이 BoW(Bag of Words)의 핵심입니다.

In [280]:
# 3개 리뷰를 각각 벡터로 변환합니다
# 어휘 순서(가나다 순): [맛있다, 배달, 빠르다, 서비스, 치킨, 피자, 좋다]

리뷰1_단어 = 리뷰1.split()
리뷰2_단어 = 리뷰2.split()
리뷰3_단어 = 리뷰3.split()

# 각 단어의 횟수를 어휘 순서에 맞춰 리스트로 만듭니다
리뷰1_BoW = [리뷰1_단어.count("맛있다"), 리뷰1_단어.count("배달"),
             리뷰1_단어.count("빠르다"), 리뷰1_단어.count("서비스"),
             리뷰1_단어.count("치킨"),   리뷰1_단어.count("피자"),
             리뷰1_단어.count("좋다")]

리뷰2_BoW = [리뷰2_단어.count("맛있다"), 리뷰2_단어.count("배달"),
             리뷰2_단어.count("빠르다"), 리뷰2_단어.count("서비스"),
             리뷰2_단어.count("치킨"),   리뷰2_단어.count("피자"),
             리뷰2_단어.count("좋다")]

리뷰3_BoW = [리뷰3_단어.count("맛있다"), 리뷰3_단어.count("배달"),
             리뷰3_단어.count("빠르다"), 리뷰3_단어.count("서비스"),
             리뷰3_단어.count("치킨"),   리뷰3_단어.count("피자"),
             리뷰3_단어.count("좋다")]

print("어휘:  [맛있다, 배달, 빠르다, 서비스, 치킨, 피자, 좋다]")
print(f"리뷰1: {리뷰1_BoW}  ← 치킨이 2!")
print(f"리뷰2: {리뷰2_BoW}")
print(f"리뷰3: {리뷰3_BoW}  ← 피자가 2!")
print()
print("⚠️  문제: 리뷰가 길수록 횟수가 무조건 커집니다 → TF로 해결!")

어휘:  [맛있다, 배달, 빠르다, 서비스, 치킨, 피자, 좋다]
리뷰1: [1, 1, 0, 0, 2, 0, 0]  ← 치킨이 2!
리뷰2: [0, 1, 1, 0, 1, 1, 0]
리뷰3: [0, 0, 0, 1, 0, 2, 1]  ← 피자가 2!

⚠️  문제: 리뷰가 길수록 횟수가 무조건 커집니다 → TF로 해결!


> **결과 해설**  
> 리뷰1에서 치킨=2, 리뷰3에서 피자=2로 가장 높습니다.  
> 하지만 이 방식에는 문제가 있습니다.  
> 만약 리뷰 길이가 2배 긴 리뷰가 있다면, 단순히 길어서 횟수가 더 높게 나올 수 있습니다.  
> **긴 리뷰가 무조건 유리한 '문서 길이 편향'** 이 발생합니다. → 다음 단계에서 해결!

---
## 📐 3단계. TF — 횟수를 비율로

```
TF = 이 단어의 횟수 / 문서 전체 단어 수   (결과: 0.0 ~ 1.0)
```

In [281]:
# TF 함수: 횟수를 전체 단어 수로 나눠 비율로 만듭니다
def TF(단어, 단어_리스트):
    횟수    = 단어_리스트.count(단어)  # 이 단어가 몇 번 나왔나
    전체_수 = len(단어_리스트)         # 문서 전체에 단어가 몇 개인가
    return 횟수 / 전체_수              # 비율 반환 (0.0 ~ 1.0)

# 리뷰1 테스트: "치킨 치킨 배달 맛있다" (총 4단어)
print(f"TF(치킨)   = 2/4 = {TF('치킨', 리뷰1_단어):.3f}  ← 전체의 절반!")
print(f"TF(배달)   = 1/4 = {TF('배달', 리뷰1_단어):.3f}")
print(f"TF(맛있다) = 1/4 = {TF('맛있다', 리뷰1_단어):.3f}")
print(f"TF(피자)   = 0/4 = {TF('피자', 리뷰1_단어):.3f}  ← 등장 안 함")

TF(치킨)   = 2/4 = 0.500  ← 전체의 절반!
TF(배달)   = 1/4 = 0.250
TF(맛있다) = 1/4 = 0.250
TF(피자)   = 0/4 = 0.000  ← 등장 안 함


> **결과 해설**  
> 리뷰1에서 '치킨'의 TF는 0.500입니다. 전체 4단어 중 절반이 '치킨'이라는 뜻입니다.  
> '배달'과 '맛있다'는 각각 0.250으로 동일하게 1번씩 등장했습니다.  
> 이처럼 TF는 **이 문서 안에서 해당 단어의 비중**을 보여줍니다.

In [282]:
# TF가 문서 길이 편향을 어떻게 해결하는지 확인
짧은 = "치킨 맛있다".split()                      # 2단어짜리 짧은 리뷰
긴   = "치킨 치킨 배달 맛있다 빠르다 좋다".split()  # 6단어짜리 긴 리뷰

print("[BoW 횟수 비교]")
print(f"  짧은 리뷰 '치킨': {짧은.count('치킨')}번")
print(f"  긴   리뷰 '치킨': {긴.count('치킨')}번  ← 긴 리뷰가 더 많아 보임 ❌")

print()
print("[TF 비율 비교]")
print(f"  짧은 리뷰 '치킨' TF: {TF('치킨', 짧은):.3f}  ({짧은.count('치킨')}/{len(짧은)})")
print(f"  긴   리뷰 '치킨' TF: {TF('치킨', 긴):.3f}  ({긴.count('치킨')}/{len(긴)})")
print()
print("  → TF 기준으로는 짧은 리뷰가 더 '치킨 중심' ✅")

[BoW 횟수 비교]
  짧은 리뷰 '치킨': 1번
  긴   리뷰 '치킨': 2번  ← 긴 리뷰가 더 많아 보임 ❌

[TF 비율 비교]
  짧은 리뷰 '치킨' TF: 0.500  (1/2)
  긴   리뷰 '치킨' TF: 0.333  (2/6)

  → TF 기준으로는 짧은 리뷰가 더 '치킨 중심' ✅


> **결과 해설**  
> - **BoW**: 긴 리뷰의 치킨이 2번으로 더 많아 보입니다. 단지 길어서입니다.  
> - **TF**: 짧은 리뷰(0.500) > 긴 리뷰(0.333) → 짧은 리뷰가 더 '치킨 중심'임을 올바르게 판단합니다.  
>
> TF는 문서 길이를 나눠서 **어떤 리뷰든 공평하게 비교**할 수 있게 해줍니다.

In [283]:
# 3개 리뷰 전체 TF 벡터 계산
# 어휘 순서: [맛있다, 배달, 빠르다, 서비스, 치킨, 피자, 좋다]
리뷰1_TF = [TF("맛있다", 리뷰1_단어), TF("배달", 리뷰1_단어),
            TF("빠르다", 리뷰1_단어), TF("서비스", 리뷰1_단어),
            TF("치킨",   리뷰1_단어), TF("피자",   리뷰1_단어),
            TF("좋다",   리뷰1_단어)]

리뷰2_TF = [TF("맛있다", 리뷰2_단어), TF("배달", 리뷰2_단어),
            TF("빠르다", 리뷰2_단어), TF("서비스", 리뷰2_단어),
            TF("치킨",   리뷰2_단어), TF("피자",   리뷰2_단어),
            TF("좋다",   리뷰2_단어)]

리뷰3_TF = [TF("맛있다", 리뷰3_단어), TF("배달", 리뷰3_단어),
            TF("빠르다", 리뷰3_단어), TF("서비스", 리뷰3_단어),
            TF("치킨",   리뷰3_단어), TF("피자",   리뷰3_단어),
            TF("좋다",   리뷰3_단어)]

print("TF 행렬  [맛있다, 배달, 빠르다, 서비스, 치킨, 피자, 좋다]")
print("-" * 55)
print(f"리뷰1: {[round(v, 3) for v in 리뷰1_TF]}")
print(f"리뷰2: {[round(v, 3) for v in 리뷰2_TF]}")
print(f"리뷰3: {[round(v, 3) for v in 리뷰3_TF]}")
print()
print("⚠️  '배달'은 리뷰1·2에서 모두 TF=0.25 → 두 리뷰를 구별 못 함 → IDF로 해결!")

TF 행렬  [맛있다, 배달, 빠르다, 서비스, 치킨, 피자, 좋다]
-------------------------------------------------------
리뷰1: [0.25, 0.25, 0.0, 0.0, 0.5, 0.0, 0.0]
리뷰2: [0.0, 0.25, 0.25, 0.0, 0.25, 0.25, 0.0]
리뷰3: [0.0, 0.0, 0.0, 0.25, 0.0, 0.5, 0.25]

⚠️  '배달'은 리뷰1·2에서 모두 TF=0.25 → 두 리뷰를 구별 못 함 → IDF로 해결!


> **결과 해설**  
> 리뷰1에서 치킨(0.500), 리뷰3에서 피자(0.500)이 가장 높아 각 리뷰의 핵심어처럼 보입니다.  
> 하지만 **'배달'은 리뷰1·2에서 모두 0.250으로 동일**합니다.  
> TF만으로는 "배달 리뷰인지"를 구별하기 어렵습니다.  
> 모든 리뷰에 흔히 등장하는 단어는 걸러낼 방법이 필요합니다. → IDF!

---
## 🔍 4단계. IDF — 희귀한 단어일수록 중요하다

```
IDF = log( 전체 문서 수 / (이 단어가 등장한 문서 수 + 1) )
```
- 모든 문서에 나오는 단어 → 분모 커짐 → IDF ≈ 0 (중요하지 않음)  
- 일부 문서에만 나오는 단어 → 분모 작음 → IDF 높음 (중요!)  
- `+1`: 분모가 0이 되는 경우를 방지 (등장 문서가 0개인 새 단어 처리)

In [284]:
# 각 단어가 3개 리뷰 중 몇 개에 등장하는지 셉니다
# "단어 in 리스트" → True(1) / False(0) 로 변환해서 더합니다

맛있다_df = (1 if "맛있다" in 리뷰1_단어 else 0) + (1 if "맛있다" in 리뷰2_단어 else 0) + (1 if "맛있다" in 리뷰3_단어 else 0)
배달_df   = (1 if "배달"   in 리뷰1_단어 else 0) + (1 if "배달"   in 리뷰2_단어 else 0) + (1 if "배달"   in 리뷰3_단어 else 0)
빠르다_df = (1 if "빠르다" in 리뷰1_단어 else 0) + (1 if "빠르다" in 리뷰2_단어 else 0) + (1 if "빠르다" in 리뷰3_단어 else 0)
서비스_df = (1 if "서비스" in 리뷰1_단어 else 0) + (1 if "서비스" in 리뷰2_단어 else 0) + (1 if "서비스" in 리뷰3_단어 else 0)
치킨_df   = (1 if "치킨"   in 리뷰1_단어 else 0) + (1 if "치킨"   in 리뷰2_단어 else 0) + (1 if "치킨"   in 리뷰3_단어 else 0)
피자_df   = (1 if "피자"   in 리뷰1_단어 else 0) + (1 if "피자"   in 리뷰2_단어 else 0) + (1 if "피자"   in 리뷰3_단어 else 0)
좋다_df   = (1 if "좋다"   in 리뷰1_단어 else 0) + (1 if "좋다"   in 리뷰2_단어 else 0) + (1 if "좋다"   in 리뷰3_단어 else 0)

print("등장 문서 수 (전체 3개 중):")
print(f"  맛있다: {맛있다_df}개  ({'■'*맛있다_df}")
print(f"  배달:   {배달_df}개  ({'■'*배달_df}")
print(f"  빠르다: {빠르다_df}개  ({'■'*빠르다_df}")
print(f"  서비스: {서비스_df}개  ({'■'*서비스_df}")
print(f"  치킨:   {치킨_df}개  ({'■'*치킨_df}")
print(f"  피자:   {피자_df}개  ({'■'*피자_df}")
print(f"  좋다:   {좋다_df}개  ({'■'*좋다_df}")

등장 문서 수 (전체 3개 중):
  맛있다: 1개  (■
  배달:   2개  (■■
  빠르다: 1개  (■
  서비스: 1개  (■
  치킨:   2개  (■■
  피자:   2개  (■■
  좋다:   1개  (■


> **결과 해설**  
> - '배달', '치킨', '피자'는 각각 2개 리뷰에 등장합니다 → 상대적으로 흔한 편  
> - '맛있다', '빠르다', '서비스', '좋다'는 1개 리뷰에만 등장합니다 → 더 희귀  
>
> 등장 문서 수가 적을수록 IDF가 높아져서 해당 리뷰를 더 잘 대표하는 단어로 평가됩니다.

In [285]:
# IDF 계산: log(전체 문서 수 / (등장 문서 수 + 1))
# math.log(): 자연로그 (밑이 e ≈ 2.718)
# 값이 클수록 희귀한 단어

N = 3   # 전체 리뷰 수

맛있다_IDF = math.log(N / (맛있다_df + 1))
배달_IDF   = math.log(N / (배달_df   + 1))
빠르다_IDF = math.log(N / (빠르다_df + 1))
서비스_IDF = math.log(N / (서비스_df + 1))
치킨_IDF   = math.log(N / (치킨_df   + 1))
피자_IDF   = math.log(N / (피자_df   + 1))
좋다_IDF   = math.log(N / (좋다_df   + 1))

print("IDF 값 (높을수록 희귀한 단어):")
print(f"  맛있다: log(3/{맛있다_df+1}) = {맛있다_IDF:.3f}  {'← 1개 리뷰에만 등장 (희귀)' if 맛있다_df==1 else ''}")
print(f"  배달:   log(3/{배달_df+1}) = {배달_IDF:.3f}  {'← 2개 리뷰에 등장 (흔함)' if 배달_df==2 else ''}")
print(f"  빠르다: log(3/{빠르다_df+1}) = {빠르다_IDF:.3f}  {'← 1개 리뷰에만 등장 (희귀)' if 빠르다_df==1 else ''}")
print(f"  서비스: log(3/{서비스_df+1}) = {서비스_IDF:.3f}  {'← 1개 리뷰에만 등장 (희귀)' if 서비스_df==1 else ''}")
print(f"  치킨:   log(3/{치킨_df+1}) = {치킨_IDF:.3f}  {'← 2개 리뷰에 등장 (흔함)' if 치킨_df==2 else ''}")
print(f"  피자:   log(3/{피자_df+1}) = {피자_IDF:.3f}  {'← 2개 리뷰에 등장 (흔함)' if 피자_df==2 else ''}")
print(f"  좋다:   log(3/{좋다_df+1}) = {좋다_IDF:.3f}  {'← 1개 리뷰에만 등장 (희귀)' if 좋다_df==1 else ''}")

IDF 값 (높을수록 희귀한 단어):
  맛있다: log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)
  배달:   log(3/3) = 0.000  ← 2개 리뷰에 등장 (흔함)
  빠르다: log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)
  서비스: log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)
  치킨:   log(3/3) = 0.000  ← 2개 리뷰에 등장 (흔함)
  피자:   log(3/3) = 0.000  ← 2개 리뷰에 등장 (흔함)
  좋다:   log(3/2) = 0.405  ← 1개 리뷰에만 등장 (희귀)


> **결과 해설**  
> 2개 리뷰에 등장하는 '배달', '치킨', '피자'는 IDF가 낮습니다.  
> 1개 리뷰에만 등장하는 '맛있다', '빠르다', '서비스', '좋다'는 IDF가 높습니다.  
>
> **IDF가 높다 = 이 단어가 등장하면 어떤 리뷰인지 특정하기 쉽다** 는 의미입니다.  
> 예를 들어 '빠르다'가 등장하면 리뷰2일 가능성이 높습니다.

---
## ⭐ 5단계. TF-IDF = TF × IDF

```
TF-IDF = TF × IDF
       = "이 문서에서 자주 나왔고" × "전체에서 희귀한 단어"
       = 이 문서의 진짜 핵심어 점수
```

In [286]:
# 리뷰1의 TF-IDF 계산
# TF × IDF: 각 단어의 TF와 IDF를 곱합니다
print(f"리뷰1: '{리뷰1}'")
print()
print(f"{'단어':8}  TF  ×  IDF  =  TF-IDF")
print("-" * 38)

맛있다_tfidf_1 = 리뷰1_TF[0] * 맛있다_IDF   # 맛있다의 TF × IDF
배달_tfidf_1   = 리뷰1_TF[1] * 배달_IDF     # 배달의 TF × IDF
빠르다_tfidf_1 = 리뷰1_TF[2] * 빠르다_IDF   # 빠르다의 TF × IDF (리뷰1엔 없으므로 0)
서비스_tfidf_1 = 리뷰1_TF[3] * 서비스_IDF
치킨_tfidf_1   = 리뷰1_TF[4] * 치킨_IDF     # 치킨: TF 높음 × IDF 적당
피자_tfidf_1   = 리뷰1_TF[5] * 피자_IDF
좋다_tfidf_1   = 리뷰1_TF[6] * 좋다_IDF

print(f"맛있다   {리뷰1_TF[0]:.3f} × {맛있다_IDF:.3f} = {맛있다_tfidf_1:.4f}")
print(f"배달     {리뷰1_TF[1]:.3f} × {배달_IDF:.3f} = {배달_tfidf_1:.4f}")
print(f"빠르다   {리뷰1_TF[2]:.3f} × {빠르다_IDF:.3f} = {빠르다_tfidf_1:.4f}")
print(f"서비스   {리뷰1_TF[3]:.3f} × {서비스_IDF:.3f} = {서비스_tfidf_1:.4f}")
print(f"치킨     {리뷰1_TF[4]:.3f} × {치킨_IDF:.3f} = {치킨_tfidf_1:.4f}  ← 최고!")
print(f"피자     {리뷰1_TF[5]:.3f} × {피자_IDF:.3f} = {피자_tfidf_1:.4f}")
print(f"좋다     {리뷰1_TF[6]:.3f} × {좋다_IDF:.3f} = {좋다_tfidf_1:.4f}")
print()
print(f"→ 리뷰1 핵심어: '치킨' ({치킨_tfidf_1:.4f})")

리뷰1: '치킨 치킨 배달 맛있다'

단어        TF  ×  IDF  =  TF-IDF
--------------------------------------
맛있다   0.250 × 0.405 = 0.1014
배달     0.250 × 0.000 = 0.0000
빠르다   0.000 × 0.405 = 0.0000
서비스   0.000 × 0.405 = 0.0000
치킨     0.500 × 0.000 = 0.0000  ← 최고!
피자     0.000 × 0.000 = 0.0000
좋다     0.000 × 0.405 = 0.0000

→ 리뷰1 핵심어: '치킨' (0.0000)


> **결과 해설**  
> '치킨'은 TF가 0.500으로 가장 높고(2번 등장), IDF도 적당히 있어서 TF-IDF가 가장 높습니다.  
> '맛있다'는 TF는 0.250이지만 IDF가 더 높아서 두 번째로 높은 점수입니다.  
> '배달'은 TF가 0.250이지만 2개 리뷰에 나오는 흔한 단어라 IDF가 낮아 점수가 낮습니다.  
>
> → **TF-IDF는 "이 리뷰에서 자주 나왔으면서, 다른 리뷰에는 별로 없는 단어"에 높은 점수를 줍니다.**

In [287]:
# 리뷰2의 TF-IDF 계산
print(f"리뷰2: '{리뷰2}'")
print()

맛있다_tfidf_2 = 리뷰2_TF[0] * 맛있다_IDF
배달_tfidf_2   = 리뷰2_TF[1] * 배달_IDF
빠르다_tfidf_2 = 리뷰2_TF[2] * 빠르다_IDF
서비스_tfidf_2 = 리뷰2_TF[3] * 서비스_IDF
치킨_tfidf_2   = 리뷰2_TF[4] * 치킨_IDF
피자_tfidf_2   = 리뷰2_TF[5] * 피자_IDF
좋다_tfidf_2   = 리뷰2_TF[6] * 좋다_IDF

# 막대 그래프로 시각화: █ 의 개수로 점수 크기를 표현
print(f"{'단어':8}  TF-IDF  막대")
print("-" * 35)
print(f"맛있다   {맛있다_tfidf_2:.4f}  {'█'*int(맛있다_tfidf_2*60)}")
print(f"배달     {배달_tfidf_2:.4f}  {'█'*int(배달_tfidf_2*60)}")
print(f"빠르다   {빠르다_tfidf_2:.4f}  {'█'*int(빠르다_tfidf_2*60)}  ← 최고!")
print(f"서비스   {서비스_tfidf_2:.4f}  {'█'*int(서비스_tfidf_2*60)}")
print(f"치킨     {치킨_tfidf_2:.4f}  {'█'*int(치킨_tfidf_2*60)}")
print(f"피자     {피자_tfidf_2:.4f}  {'█'*int(피자_tfidf_2*60)}  ← 최고!")
print(f"좋다     {좋다_tfidf_2:.4f}  {'█'*int(좋다_tfidf_2*60)}")
print()
print(f"→ 리뷰2 핵심어: '빠르다'({빠르다_tfidf_2:.4f}), '피자'({피자_tfidf_2:.4f})")

리뷰2: '치킨 피자 배달 빠르다'

단어        TF-IDF  막대
-----------------------------------
맛있다   0.0000  
배달     0.0000  
빠르다   0.1014  ██████  ← 최고!
서비스   0.0000  
치킨     0.0000  
피자     0.0000    ← 최고!
좋다     0.0000  

→ 리뷰2 핵심어: '빠르다'(0.1014), '피자'(0.0000)


> **결과 해설**  
> 리뷰2 `"치킨 피자 배달 빠르다"` 에서 '빠르다'와 '피자'가 공동 최고점입니다.  
> '치킨'도 있지만 2개 리뷰에 흔히 나와서 IDF가 낮아 점수가 낮습니다.  
> '빠르다'는 오직 리뷰2에만 등장하기 때문에 IDF가 높아서 핵심어가 됩니다.  
>
> → TF-IDF는 단순히 많이 나온 단어가 아니라 **이 리뷰만의 특징을 잡아냅니다.**

In [288]:
# 3개 리뷰 핵심어 최종 정리
맛있다_tfidf_3 = 리뷰3_TF[0] * 맛있다_IDF
피자_tfidf_3   = 리뷰3_TF[5] * 피자_IDF

print("=" * 40)
print("📊 TF-IDF 핵심어 요약")
print("=" * 40)
print(f"리뷰1: '{리뷰1}'")
print(f"  → 핵심어: '치킨' ({치킨_tfidf_1:.4f})")
print()
print(f"리뷰2: '{리뷰2}'")
print(f"  → 핵심어: '빠르다'({빠르다_tfidf_2:.4f}), '피자'({피자_tfidf_2:.4f})")
print()
print(f"리뷰3: '{리뷰3}'")
print(f"  → 핵심어: '피자' ({피자_tfidf_3:.4f})")
print()
print("직접 계산 완료! ✅  이제 sklearn으로 같은 결과를 3줄로 →")

📊 TF-IDF 핵심어 요약
리뷰1: '치킨 치킨 배달 맛있다'
  → 핵심어: '치킨' (0.0000)

리뷰2: '치킨 피자 배달 빠르다'
  → 핵심어: '빠르다'(0.1014), '피자'(0.0000)

리뷰3: '피자 피자 서비스 좋다'
  → 핵심어: '피자' (0.0000)

직접 계산 완료! ✅  이제 sklearn으로 같은 결과를 3줄로 →


> **결과 해설**  
> - 리뷰1: '치킨' — 치킨 위주의 리뷰임을 정확히 포착했습니다  
> - 리뷰2: '빠르다', '피자' — 배달 속도와 피자에 대한 리뷰임을 나타냅니다  
> - 리뷰3: '피자' — 피자 위주의 리뷰로 분류됩니다  
>
> 직접 손으로 계산해서 원리를 확인했습니다. 이제 sklearn이 이걸 자동으로 처리하는 걸 봅시다.

---
## 🛠️ 6단계. sklearn — 3줄로 끝내기

원리를 이해했으니, 실전에서는 sklearn이 자동으로 처리합니다.

In [289]:
# TfidfVectorizer: TF-IDF 계산을 자동으로 해주는 sklearn 도구
vectorizer = TfidfVectorizer()

# fit_transform(): 두 가지를 동시에 수행
#   fit     → 어휘 목록 만들기 + 각 단어의 IDF 계산 (학습)
#   transform → 각 리뷰를 TF-IDF 벡터로 변환
tfidf_행렬 = vectorizer.fit_transform([리뷰1, 리뷰2, 리뷰3])

# get_feature_names_out(): 학습된 어휘 목록 확인
어휘 = vectorizer.get_feature_names_out()

print("어휘:", 어휘)
print("행렬 크기:", tfidf_행렬.shape, " ← (리뷰 수 × 단어 수)")

어휘: ['맛있다' '배달' '빠르다' '서비스' '좋다' '치킨' '피자']
행렬 크기: (3, 7)  ← (리뷰 수 × 단어 수)


> **결과 해설**  
> 어휘에 7개 단어가 있고, 행렬 크기가 `(3, 7)` 입니다.  
> 3개 리뷰 × 7개 단어의 TF-IDF 점수 행렬이 만들어졌다는 뜻입니다.

In [290]:
# .toarray(): 희소 행렬(메모리 절약형) → 일반 숫자 배열로 변환
배열 = tfidf_행렬.toarray()

print("TF-IDF 행렬:")
print(f"{'':8}" + "  ".join(f"{w:7}" for w in 어휘))
print("-" * 65)
print(f"리뷰1:  " + "  ".join(f"{v:.4f}" for v in 배열[0]))
print(f"리뷰2:  " + "  ".join(f"{v:.4f}" for v in 배열[1]))
print(f"리뷰3:  " + "  ".join(f"{v:.4f}" for v in 배열[2]))
print()
print("※ 직접 계산값과 약간 다릅니다.")
print("  sklearn은 IDF 공식이 조금 다르고, L2 정규화(벡터 크기를 1로 조정)를 추가합니다.")
print("  개념은 동일 — 실전에서는 항상 sklearn을 사용합니다.")

TF-IDF 행렬:
        맛있다      배달       빠르다      서비스      좋다       치킨       피자     
-----------------------------------------------------------------
리뷰1:  0.5069  0.3855  0.0000  0.0000  0.0000  0.7710  0.0000
리뷰2:  0.0000  0.4599  0.6047  0.0000  0.0000  0.4599  0.4599
리뷰3:  0.0000  0.0000  0.0000  0.4815  0.4815  0.0000  0.7324

※ 직접 계산값과 약간 다릅니다.
  sklearn은 IDF 공식이 조금 다르고, L2 정규화(벡터 크기를 1로 조정)를 추가합니다.
  개념은 동일 — 실전에서는 항상 sklearn을 사용합니다.


> **결과 해설**  
> 직접 계산한 값과 조금 다른 이유는 sklearn이 수치 안정성을 위해 IDF 공식을 약간 다르게 쓰기 때문입니다.  
> 핵심어(가장 높은 값)는 동일하게 나옵니다.  
> 실전 프로젝트에서는 항상 sklearn을 사용합니다 — 더 빠르고 안정적입니다.

In [291]:
# .argmax(): 배열에서 가장 큰 값의 인덱스를 반환
# 그 인덱스로 어휘에서 단어를 찾아냅니다

리뷰1_핵심 = 어휘[배열[0].argmax()]   # 리뷰1에서 TF-IDF가 가장 높은 단어
리뷰2_핵심 = 어휘[배열[1].argmax()]   # 리뷰2에서 TF-IDF가 가장 높은 단어
리뷰3_핵심 = 어휘[배열[2].argmax()]   # 리뷰3에서 TF-IDF가 가장 높은 단어

print(f"리뷰1 핵심어: '{리뷰1_핵심}'")
print(f"리뷰2 핵심어: '{리뷰2_핵심}'")
print(f"리뷰3 핵심어: '{리뷰3_핵심}'")

리뷰1 핵심어: '치킨'
리뷰2 핵심어: '빠르다'
리뷰3 핵심어: '피자'


> **결과 해설**  
> 직접 계산한 결과와 핵심어가 동일하게 나왔다면 성공입니다!  
> 이제 5단계에 걸쳐 손으로 계산한 것을 sklearn 3줄이 자동으로 처리한 것입니다.

---
## 🍽️ 7단계. 실제 리뷰에 적용하기

2주차 전처리 + 3주차 TF-IDF를 실제 리뷰에 연결합니다.

```
실제 리뷰 → 전처리() → ' '.join() → TfidfVectorizer
```

In [292]:
# 노이즈가 포함된 실제 배달 리뷰
원본1 = "ㅋㅋ 치킨이 진짜 바삭하고 맛있어요! 소스도 최고"
원본2 = "배달이 너무 느려서 음식이 다 식었어요ㅠ"
원본3 = "피자 도우가 쫄깃하고 치즈가 엄청 많아요 가성비 최고"

# 전처리: 노이즈 제거 + 명사 추출
처리1 = 전처리(원본1)
처리2 = 전처리(원본2)
처리3 = 전처리(원본3)

print("전처리 결과:")
print(f"  리뷰1: {처리1}")
print(f"  리뷰2: {처리2}")
print(f"  리뷰3: {처리3}")

전처리 결과:
  리뷰1: ['치킨', '진짜', '바삭', '소스', '최고']
  리뷰2: ['배달', '음식']
  리뷰3: ['피자', '도우', '쫄깃', '치즈', '가성', '최고']


> **결과 해설**  
> 'ㅋㅋ', '!', 'ㅠ' 같은 노이즈가 제거되고, '진짜', '너무' 같은 부사도 사라졌습니다.  
> '바삭하고' → '바삭', '식었어요' → '음식' 처럼 형태소 분석으로 핵심 명사만 남습니다.

In [293]:
# ' '.join(리스트): 리스트를 공백으로 연결해 문자열로 만듭니다
# sklearn TfidfVectorizer는 문자열을 입력으로 받기 때문에 변환이 필요합니다
문자열1 = ' '.join(처리1)   # ['치킨', '바삭'] → '치킨 바삭'
문자열2 = ' '.join(처리2)
문자열3 = ' '.join(처리3)

print("sklearn 입력 형식:")
print(f"  리뷰1: '{문자열1}'")
print(f"  리뷰2: '{문자열2}'")
print(f"  리뷰3: '{문자열3}'")

sklearn 입력 형식:
  리뷰1: '치킨 진짜 바삭 소스 최고'
  리뷰2: '배달 음식'
  리뷰3: '피자 도우 쫄깃 치즈 가성 최고'


In [294]:
# TF-IDF 생성 + 핵심어 출력
실제_vec   = TfidfVectorizer()
실제_tfidf = 실제_vec.fit_transform([문자열1, 문자열2, 문자열3])
실제_어휘  = 실제_vec.get_feature_names_out()
실제_배열  = 실제_tfidf.toarray()

# argmax()로 각 리뷰에서 가장 높은 TF-IDF 단어를 찾습니다
핵심1 = 실제_어휘[실제_배열[0].argmax()]
핵심2 = 실제_어휘[실제_배열[1].argmax()]
핵심3 = 실제_어휘[실제_배열[2].argmax()]

print(f"리뷰1: '{원본1}'")
print(f"  → 핵심어: '{핵심1}'")
print()
print(f"리뷰2: '{원본2}'")
print(f"  → 핵심어: '{핵심2}'")
print()
print(f"리뷰3: '{원본3}'")
print(f"  → 핵심어: '{핵심3}'")

리뷰1: 'ㅋㅋ 치킨이 진짜 바삭하고 맛있어요! 소스도 최고'
  → 핵심어: '바삭'

리뷰2: '배달이 너무 느려서 음식이 다 식었어요ㅠ'
  → 핵심어: '배달'

리뷰3: '피자 도우가 쫄깃하고 치즈가 엄청 많아요 가성비 최고'
  → 핵심어: '가성'


> **결과 해설**  
> 리뷰1은 치킨의 식감('바삭')이, 리뷰2는 배달 지연이, 리뷰3은 피자 관련 단어가 핵심어로 추출됩니다.  
> 사람이 읽고 "이 리뷰는 치킨 리뷰다"라고 판단하는 것과 유사하게 작동합니다.  
> 이 핵심어들이 다음 주 머신러닝 분류기의 입력 특성이 됩니다.

---
## 🔗 8단계. 코사인 유사도 — 얼마나 비슷한가?

```
코사인 유사도 = 두 벡터가 얼마나 같은 방향을 가리키는가
              0.0 (완전히 다름)  ~  1.0 (완전히 같음)
```
길이(문서 분량)는 무시하고 **방향(어떤 단어를 쓰는가)** 만 비교합니다.

In [295]:
# cosine_similarity(): TF-IDF 행렬을 받아 모든 문서 쌍의 유사도를 계산
# 결과는 N×N 행렬: [i][j] = 리뷰i와 리뷰j의 유사도
유사도 = cosine_similarity(실제_tfidf)

print("리뷰 간 유사도:")
print()
# 막대 그래프: █ 수로 유사도 크기를 직관적으로 표현
print(f"  리뷰1 ↔ 리뷰2: {유사도[0][1]:.4f}  {'█'*int(유사도[0][1]*50)}")
print(f"  리뷰1 ↔ 리뷰3: {유사도[0][2]:.4f}  {'█'*int(유사도[0][2]*50)}")
print(f"  리뷰2 ↔ 리뷰3: {유사도[1][2]:.4f}  {'█'*int(유사도[1][2]*50)}")
print()
print(f"리뷰1: '{원본1}'")
print(f"리뷰2: '{원본2}'")
print(f"리뷰3: '{원본3}'")

리뷰 간 유사도:

  리뷰1 ↔ 리뷰2: 0.0000  
  리뷰1 ↔ 리뷰3: 0.1145  █████
  리뷰2 ↔ 리뷰3: 0.0000  

리뷰1: 'ㅋㅋ 치킨이 진짜 바삭하고 맛있어요! 소스도 최고'
리뷰2: '배달이 너무 느려서 음식이 다 식었어요ㅠ'
리뷰3: '피자 도우가 쫄깃하고 치즈가 엄청 많아요 가성비 최고'


> **결과 해설**  
> 리뷰1(치킨 바삭), 리뷰2(배달 느림), 리뷰3(피자 쫄깃)은 주제가 모두 달라서 유사도가 낮게 나옵니다.  
> 만약 리뷰1과 비슷한 "치킨이 바삭해서 맛있었어요"가 있다면 유사도가 높게 나올 것입니다.  
>
> 유사도가 **0.3 이상**이면 비슷한 리뷰, **0.1 미만**이면 전혀 다른 리뷰로 볼 수 있습니다.  
> (기준은 데이터마다 다를 수 있습니다)

In [296]:
# 가장 비슷한 리뷰 쌍 찾기
유사도_12 = 유사도[0][1]   # 리뷰1과 리뷰2의 유사도
유사도_13 = 유사도[0][2]   # 리뷰1과 리뷰3의 유사도
유사도_23 = 유사도[1][2]   # 리뷰2와 리뷰3의 유사도

가장_높음 = max(유사도_12, 유사도_13, 유사도_23)

# 가장 높은 유사도가 어떤 쌍인지 찾기
if 가장_높음 == 유사도_12:
    print(f"가장 비슷한 쌍: 리뷰1 ↔ 리뷰2  (유사도: {유사도_12:.4f})")
elif 가장_높음 == 유사도_13:
    print(f"가장 비슷한 쌍: 리뷰1 ↔ 리뷰3  (유사도: {유사도_13:.4f})")
else:
    print(f"가장 비슷한 쌍: 리뷰2 ↔ 리뷰3  (유사도: {유사도_23:.4f})")

가장 비슷한 쌍: 리뷰1 ↔ 리뷰3  (유사도: 0.1145)


> **결과 해설**  
> 3개 리뷰 중 가장 비슷한 쌍을 자동으로 찾아냅니다.  
> 이 원리가 **검색 엔진**(질문과 가장 비슷한 문서 찾기)과  
> **RAG 챗봇**(질문에 관련된 지식 문서 찾기)의 핵심입니다.

---
## 🏆 9단계. 도전 과제 — 뉴스 기사 분류

**목표:** 같은 주제 기사끼리 유사도가 높게 나오면 성공! 🎯

| 기사 | 주제 |
|------|------|
| 기사1, 기사2 | 스마트폰 |
| 기사3, 기사4 | 스포츠 |
| 기사5, 기사6 | AI |

In [297]:
# 뉴스 기사 6개
기사1 = "삼성전자가 국내 스마트폰 시장에서 압도적 1위를 지키고 있는 상황에서 애플이 3년 연속 점유율을 끌어올려 빠르게 추격하고 있는 것으로 나타났다."
기사2 = "스마트폰 시장에서 배터리 용량이 눈에 띄게 변화하고 있습니다. 현재 400달러 내외의 가격대에서 판매되는 많은 스마트폰에 최대 10,000mAh의 배터리가 탑재되고 있습니다. "
기사3 = "미국프로야구 메이저리그(MLB)에서 활약하는 이정후(샌프란시스코 자이언츠)가 시범 경기에서 3타수 2안타 활약을 펼쳤다."
기사4 = "야구팬들 사이에서 '시범경기 성적은 믿을 게 못 된다'는 말은 오랜 격언처럼 통한다.2001년부터 2025년까지 KBO리그 순위표를 통계로 분석해 본 결과, 이 속설은 완벽한 사실로 확인됐다."
기사5 = "인공지능(AI) 기반 일자리 매칭 서비스가 작년 하루 평균 57명의 구직자를 취업시킨 것으로 나타났다. AI를 활용한 기업의 구인 공고는 일반 구인 공고보다 평균 입사 지원자 수가 41% 많았다."
기사6 = "평택시와 평택산업진흥원이 평택 중소·중견기업의 현안 해결과 인공지능 전환 촉진을 위해 ‘2026년 AI 수요기반 실증사업’ 수요기업 모집을 시작으로 제조 인공지능 생태계 조성에 나선다고 3월 13일 밝혔다."

print("기사1 [스마트폰]:", 기사1[:40], "...")
print("기사2 [스마트폰]:", 기사2[:40], "...")
print("기사3 [스포츠]  :", 기사3[:40], "...")
print("기사4 [스포츠]  :", 기사4[:40], "...")
print("기사5 [AI]     :", 기사5[:40], "...")
print("기사6 [AI]     :", 기사6[:40], "...")

기사1 [스마트폰]: 삼성전자가 국내 스마트폰 시장에서 압도적 1위를 지키고 있는 상황에서 애 ...
기사2 [스마트폰]: 스마트폰 시장에서 배터리 용량이 눈에 띄게 변화하고 있습니다. 현재 40 ...
기사3 [스포츠]  : 미국프로야구 메이저리그(MLB)에서 활약하는 이정후(샌프란시스코 자이언츠 ...
기사4 [스포츠]  : 야구팬들 사이에서 '시범경기 성적은 믿을 게 못 된다'는 말은 오랜 격언 ...
기사5 [AI]     : 인공지능(AI) 기반 일자리 매칭 서비스가 작년 하루 평균 57명의 구직 ...
기사6 [AI]     : 평택시와 평택산업진흥원이 평택 중소·중견기업의 현안 해결과 인공지능 전환 ...


In [298]:
# STEP 1: 전처리 — 6개 기사 각각 명사 추출
처리_기사1 = 전처리(기사1)
처리_기사2 = 전처리(기사2)
처리_기사3 = 전처리(기사3)
처리_기사4 = 전처리(기사4)
처리_기사5 = 전처리(기사5)
처리_기사6 = 전처리(기사6)

print("전처리 결과 (명사만 추출):")
print(f"  기사1: {처리_기사1}")
print(f"  기사2: {처리_기사2}")
print(f"  기사3: {처리_기사3}")
print(f"  기사4: {처리_기사4}")
print(f"  기사5: {처리_기사5}")
print(f"  기사6: {처리_기사6}")

전처리 결과 (명사만 추출):
  기사1: ['전자', '국내', '스마트폰', '시장', '압도', '상황', '애플', '연속', '점유', '추격']
  기사2: ['스마트폰', '시장', '배터리', '용량', '변화', '현재', '내외', '가격', '판매', '스마트폰', '최대', '배터리', '탑재']
  기사3: ['미국', '프로야구', '메이저리그', '활약', '이정후', '샌프란시스코', '자이언츠', '시범', '경기', '타수', '안타', '활약']
  기사4: ['야구', '사이', '시범', '경기', '성적', '격언', '통한', '리그', '순위표', '통계', '분석', '결과', '사실', '확인']
  기사5: ['인공', '지능', '기반', '일자리', '매칭', '서비스', '작년', '하루', '평균', '명의', '구직', '취업', '활용', '기업', '구인', '공고', '일반', '구인', '공고', '평균', '입사', '지원', '수가']
  기사6: ['평택시', '평택', '산업', '진흥', '평택', '중소', '중견', '기업', '현안', '해결', '인공', '지능', '전환', '촉진', '위해', '수요', '기반', '실증', '사업', '수요', '기업', '모집', '시작', '제조', '인공', '지능', '생태계', '조성']


> **결과 해설**  
> 기사1·2에는 '갤럭시', '스마트폰', '카메라', '배터리' 같은 공통 단어가 있을 것입니다.  
> 기사3·4에는 '야구', '축구', '월드컵' 같은 스포츠 단어가, 기사5·6에는 'AI', '인공지능'이 나올 것입니다.  
> 이 공통 단어들이 유사도 계산에서 핵심 역할을 합니다.

In [299]:
# STEP 2: TF-IDF 행렬 생성
# 전처리 결과(리스트)를 ' '.join()으로 문자열로 변환 후 vectorizer에 전달
기사_vec = TfidfVectorizer()
기사_tfidf = 기사_vec.fit_transform([
    ' '.join(처리_기사1), ' '.join(처리_기사2),
    ' '.join(처리_기사3), ' '.join(처리_기사4),
    ' '.join(처리_기사5), ' '.join(처리_기사6)
])
기사_어휘 = 기사_vec.get_feature_names_out()
기사_배열 = 기사_tfidf.toarray()

print(f"전체 어휘: {len(기사_어휘)}개")
print(f"행렬 크기: {기사_tfidf.shape}  ← (기사 6개 × 단어 수)")

전체 어휘: 81개
행렬 크기: (6, 81)  ← (기사 6개 × 단어 수)


In [300]:
# STEP 3: 기사별 핵심어 (.argmax()로 TF-IDF 최고 단어 추출)
print("기사별 핵심어:")
print(f"  기사1 [스마트폰]: '{기사_어휘[기사_배열[0].argmax()]}'  ({기사_배열[0].max():.3f})")
print(f"  기사2 [스마트폰]: '{기사_어휘[기사_배열[1].argmax()]}'  ({기사_배열[1].max():.3f})")
print(f"  기사3 [스포츠]:   '{기사_어휘[기사_배열[2].argmax()]}'  ({기사_배열[2].max():.3f})")
print(f"  기사4 [스포츠]:   '{기사_어휘[기사_배열[3].argmax()]}'  ({기사_배열[3].max():.3f})")
print(f"  기사5 [AI]:       '{기사_어휘[기사_배열[4].argmax()]}'  ({기사_배열[4].max():.3f})")
print(f"  기사6 [AI]:       '{기사_어휘[기사_배열[5].argmax()]}'  ({기사_배열[5].max():.3f})")

기사별 핵심어:
  기사1 [스마트폰]: '국내'  (0.327)
  기사2 [스마트폰]: '배터리'  (0.510)
  기사3 [스포츠]:   '활약'  (0.547)
  기사4 [스포츠]:   '격언'  (0.274)
  기사5 [AI]:       '공고'  (0.380)
  기사6 [AI]:       '수요'  (0.344)


> **결과 해설**  
> 각 기사의 핵심어가 주제와 관련 있는 단어로 나왔다면 TF-IDF가 잘 작동하는 것입니다.  
> 예를 들어 기사1의 핵심어가 '갤럭시'나 '스마트폰', 기사3이 '한국시리즈'나 '투수'라면 성공입니다.

In [301]:
# STEP 4: 유사도 행렬 — 같은 주제 기사끼리 높게 나오면 성공!
기사_유사도 = cosine_similarity(기사_tfidf)

print("유사도 확인:")
print()
# 같은 주제 쌍: 높아야 함
print(f"  [같은 주제] 스마트폰: 기사1 ↔ 기사2 = {기사_유사도[0][1]:.3f}  {'✅ 높음!' if 기사_유사도[0][1] > 0.1 else '🤔 낮음'}")
print(f"  [같은 주제] 스포츠:   기사3 ↔ 기사4 = {기사_유사도[2][3]:.3f}  {'✅ 높음!' if 기사_유사도[2][3] > 0.1 else '🤔 낮음'}")
print(f"  [같은 주제] AI:       기사5 ↔ 기사6 = {기사_유사도[4][5]:.3f}  {'✅ 높음!' if 기사_유사도[4][5] > 0.1 else '🤔 낮음'}")
print()
# 다른 주제 쌍: 낮아야 함
print(f"  [다른 주제] 기사1(스마트폰) ↔ 기사3(스포츠) = {기사_유사도[0][2]:.3f}  {'✅ 낮음!' if 기사_유사도[0][2] < 0.1 else '🤔 높음'}")
print(f"  [다른 주제] 기사1(스마트폰) ↔ 기사5(AI)     = {기사_유사도[0][4]:.3f}  {'✅ 낮음!' if 기사_유사도[0][4] < 0.1 else '🤔 높음'}")
print()
print("같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! 🎯")

유사도 확인:

  [같은 주제] 스마트폰: 기사1 ↔ 기사2 = 0.168  ✅ 높음!
  [같은 주제] 스포츠:   기사3 ↔ 기사4 = 0.101  ✅ 높음!
  [같은 주제] AI:       기사5 ↔ 기사6 = 0.154  ✅ 높음!

  [다른 주제] 기사1(스마트폰) ↔ 기사3(스포츠) = 0.000  ✅ 낮음!
  [다른 주제] 기사1(스마트폰) ↔ 기사5(AI)     = 0.000  ✅ 낮음!

같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! 🎯


> **결과 해설**  
> - **같은 주제 기사 쌍**의 유사도가 높다면 → TF-IDF가 주제를 잘 포착한 것입니다  
> - **다른 주제 기사 쌍**의 유사도가 낮다면 → 불필요한 공통 단어가 잘 걸러진 것입니다  
>
> 이 원리로 검색 엔진은 "관련 기사 더 보기"를, 유튜브는 "다음으로 볼 영상"을 추천합니다.  
> 그리고 이것이 12~16주차에서 만들 **RAG 챗봇의 핵심 원리**입니다!

---
## 📌 전체 흐름 정리

```
텍스트
  ↓ 전처리()              노이즈 제거, 명사 추출
단어 리스트
  ↓ .count()              각 단어가 몇 번?
BoW (횟수)
  ↓ ÷ 전체 단어 수        문서 길이에 공평하게
TF (비율, 0.0~1.0)
  ↓ × log(N / df)         흔한 단어 억제
TF-IDF (핵심어 점수)
  ↓ cosine_similarity()
유사도 행렬               어떤 문서가 비슷한가?
```

### 핵심 코드
```python
vectorizer = TfidfVectorizer()
tfidf      = vectorizer.fit_transform(["단어1 단어2", "단어3 단어4"])
유사도     = cosine_similarity(tfidf)
```

### 다음 주 (4주차)
TF-IDF 벡터 → **머신러닝 분류기** → `"긍정"` / `"부정"` 🚀